In [8]:
import pandas as pd 

mercadoLivre = pd.read_excel("onibus_mercadolivre.xlsx")

In [9]:
import re

gabarito = pd.read_excel("Gabarito.xlsx")

def normalizar(texto):
    if pd.isna(texto):
        return ""
    return re.sub(r'\s+', ' ', str(texto).upper().strip())

titulos_norm = mercadoLivre['Título'].apply(normalizar)
gabarito['_marca_norm'] = gabarito['MARCA'].apply(normalizar)
gabarito['_modelo_norm'] = gabarito['MODELO'].apply(normalizar)

def encontrar_match(titulo_norm):
    melhor_score = -1
    melhor_marca = ""
    melhor_modelo = ""

    for _, row in gabarito.iterrows():
        marca = row['_marca_norm']
        modelo = row['_modelo_norm']
        marca_ok = bool(marca) and marca in titulo_norm
        modelo_ok = bool(modelo) and modelo in titulo_norm

        if marca_ok and modelo_ok:
            score = len(marca) + len(modelo) 
            if score > melhor_score:
                melhor_score = score
                melhor_marca = row['MARCA']
                melhor_modelo = row['MODELO']

    if not melhor_marca:
        marcas_vistas = set()
        for _, row in gabarito.iterrows():
            marca = row['_marca_norm']
            if bool(marca) and marca not in marcas_vistas and marca in titulo_norm:
                melhor_marca = row['MARCA']
                marcas_vistas.add(marca)
                break

    return melhor_marca, melhor_modelo

resultados = titulos_norm.apply(encontrar_match)

mercadoLivre['Marca'] = [r[0] for r in resultados]
mercadoLivre['Modelo'] = [r[1] for r in resultados]

gabarito.drop(columns=['_marca_norm', '_modelo_norm'], inplace=True)

total = len(mercadoLivre)
marca_preenchidos = (mercadoLivre['Marca'].fillna('') != '').sum()
modelo_preenchidos = (mercadoLivre['Modelo'].fillna('') != '').sum()

print(f"[DEBUG 1] Total de registros: {total}")
print(f"[DEBUG 2] Marca  — Preenchidos: {marca_preenchidos} | Vazios: {total - marca_preenchidos}")
print(f"[DEBUG 3] Modelo — Preenchidos: {modelo_preenchidos} | Vazios: {total - modelo_preenchidos}")


[DEBUG 1] Total de registros: 2016
[DEBUG 2] Marca  — Preenchidos: 1829 | Vazios: 187
[DEBUG 3] Modelo — Preenchidos: 1630 | Vazios: 386


In [10]:
mercadoLivre.to_excel("onibus_mercadolivre_tratado.xlsx", index=False)
print("Base salva em: onibus_mercadolivre_tratado.xlsx")

Base salva em: onibus_mercadolivre_tratado.xlsx
